# Geometry, mesh, and hybrid boundaries

CPU-only introduction to the public `fdtdmesh` workflow. Shapes use SI metres; the automatic domain uses the documented wavelength-fraction defaults.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/fdtdmesh').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
import matplotlib.pyplot as plt
import fdtdmesh as fd
from fdtdmesh.boundary import BoundaryPolicy
OUT = Path('artifacts/notebooks')
OUT.mkdir(parents=True, exist_ok=True)


## Exact geometry and automatic layout

A centred PEC circle with a smaller air cut demonstrates material overlays without editing the geometry into a raster.

In [ ]:
lam = fd.C0 / 1e9
sim = fd.Simulation(fmin=0.9e9, fmax=1.1e9, solver=fd.SolverSettings(dft_bins=1), boundary=BoundaryPolicy(mode='hybrid'))
sim.add_circle((3*lam, 3*lam), 0.48*lam, name='centred-circle')
sim.add_circle((3*lam, 3*lam), 0.12*lam, material='air', name='air-cut')
print('automatic layout:', sim.summary()['layout'])
sim.apply_mesh('geometry_aware', cells=(120, 120))
fig = sim.plot_geometry(mesh=True, units='wavelength', figsize=(16, 14))
fig.savefig(OUT / '01_geometry_and_grid.png', dpi=120, bbox_inches='tight')


## Discretization and source

The large figure exposes local cut faces, enlarged donor pairs, and any hybrid staircase fallback.

In [ ]:
fig = sim.plot_discretization(units='wavelength', show_fallback=True, figsize=(16, 14))
fig.savefig(OUT / '02_discretization_fallbacks.png', dpi=120, bbox_inches='tight')
sim.plot_source().savefig(OUT / '03_source.png', dpi=120, bbox_inches='tight')
print('mesh diagnostics:', sim.summary())


The same public calls support strict conformal validation when a study needs an exact grid. Hybrid mode changes local edge coefficients where a conformal cut cannot be resolved; it does not alter the continuous geometry.